# 创建作业文件夹
```
桌面
│
└── docker-homework
          | ----docker-compose.yml
          | ----Dockerfile
```
# 编辑 Dockerfile
```dockerfile
FROM docker.m.daocloud.io/ollama/ollama:latest

EXPOSE 11434

CMD ["serve"]
```
# 编辑 docker-compose.yml
```yaml
services:
  ollama:
    build: .
    container_name: qwen3_service

    ports:
      - "11434:11434"

    volumes:
      - ollama_data:/root/.ollama

volumes:
  ollama_data:
```
# PowerShell处操作
```powershell
cd Desktop\docker-homework
```
# 构建容器

```powershell
docker compose up -d
```
下载镜像成功后
```powershell
docker ps
```
应该看到：
```text
qwen3_service
```
# 进入容器
```powershell
docker exec -it qwen3_service bash
```
# 启动 Ollama 服务
容器里执行：
```bash
ollama serve
```
再开一个新的 PowerShell 窗口。
# 下载模型
在新的 PowerShell 中执行：
```powershell
docker exec -it qwen3_service ollama pull qwen3:0.6b
```
等待下载完成。
查看：
```powershell
docker exec -it qwen3_service ollama list
```
如果看到：
```text
qwen3:0.6b
```
说明模型下载成功。
# 测试模型
PowerShell：
```powershell
curl http://localhost:11434/api/tags
```
如果返回：

```json
{
  "models":[
    {
      "name":"qwen3:0.6b"
    }
  ]
}
```
说明服务已经对外开放。

# 创建测试程序

桌面再建一个文件：

```text
test.py
```

写入：

```python
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:0.6b",
        "prompt": "你好",
        "stream": False
    }
)

print(response.json()["response"])
```



# 安装 requests

PowerShell：

```powershell
pip install requests
```

# 运行测试

```powershell
cd Desktop
python test.py
```
输出类似：
```text
你好！有什么可以帮助你的吗？（具体见上传的图片）
```
这就证明：

* Docker里的模型正常运行
* Docker外面的Python程序成功调用


**Python程序可以调用，说明API服务完全正常**

## 在容器中使用千问
用 `ollama` 命令拉取/运行模型就好：

```bash
# 在容器内执行，拉取并运行模型（比如 qwen3）
ollama pull qwen3

# 或者直接运行
ollama run qwen3
```
## 验证服务是否正常

```bash
# 在容器内测试
curl http://localhost:11434

# 或在 Windows PowerShell 里测试
curl http://localhost:11434
```
返回 `Ollama is running` 就说明一切正常。
在容器中使用千问的场景可见截图

## Cherry Studio 配置步骤

**1. 打开设置 → 找到模型服务**

左下角齿轮图标 → **模型服务（Model Provider）**

**2. 添加 Ollama 服务**

找到 Ollama 选项（或添加 OpenAI 兼容服务），填写：

| 配置项 | 填写值 |
|--------|--------|
| API 地址 | `http://localhost:11434` |
| API Key | `ollama`（随便填） |

**3. 添加模型**

在模型列表里手动添加：
```
qwen3:0.6b
```

**4. 点击"检测"或"连接测试"**

显示绿色连接成功即可。

**5. 新建对话，选择 qwen3:0.6b 模型，发送消息验证。**

**6.验证api可用**
在容器内：
```bash
apt update && apt install -y curl
```
```bash
curl -X POST http://localhost:11434/api/chat -H "Content-Type: application/json" -d '{"model":"qwen3:0.6b","messages":[{"role":"user","content":"你好"}]}'
```
有流式返回就说明 API 完全正常，Cherry Studio 可以正常调用了。